# 8. Revenue-at-Risk & ROI Simulation

## Objective

Translate churn probabilities into business impact.


In [1]:
import sys
import os
sys.path.append(os.path.abspath(".."))

import pandas as pd
import numpy as np

from src.config import PROCESSED_DATA_PATH
from src.modeling import prepare_data, train_logistic

In [2]:
path = os.path.join(PROCESSED_DATA_PATH, "modeling_dataset.csv")
df = pd.read_csv(path)

X_train, X_test, y_train, y_test, scaler = prepare_data(df)

model = train_logistic(X_train, y_train)

# Predicted probabilities
probs = model.predict_proba(X_test)[:, 1]

results_df = pd.DataFrame({
    "CustomerID": df.iloc[y_test.index]["CustomerID"].values,
    "Churn_Prob": probs,
    "Actual_Churn": y_test.values,
    "Monetary": df.iloc[y_test.index]["Monetary"].values
})

## Phase 1: Probability-Only Targeting

Ranking customers purely by churn probability resulted in:

- Low revenue concentration
- Negative campaign ROI


In [3]:
# Ranking Customer by Risk
results_df = results_df.sort_values("Churn_Prob", ascending=False)

In [5]:
# Selecting Top 15% High-Risk Customers
top_percent = 0.15
top_n = int(len(results_df) * top_percent)

high_risk = results_df.head(top_n)

In [6]:
revenue_at_risk = high_risk["Monetary"].sum()

print("Revenue at Risk (Top 15%):", revenue_at_risk)

Revenue at Risk (Top 15%): 67875.89


In [7]:
total_revenue = results_df["Monetary"].sum()

print("Percentage of Revenue at Risk:",
      revenue_at_risk / total_revenue)

Percentage of Revenue at Risk: 0.020561817381852043


In [8]:
# Retention Simulation
success_rate = 0.20
cost_per_customer = 200

customers_targeted = len(high_risk)

saved_revenue = revenue_at_risk * success_rate
campaign_cost = customers_targeted * cost_per_customer

roi = (saved_revenue - campaign_cost) / campaign_cost

print("Saved Revenue:", saved_revenue)
print("Campaign Cost:", campaign_cost)
print("ROI:", roi)

Saved Revenue: 13575.178
Campaign Cost: 31400
ROI: -0.5676694904458599


### Insight:
High churn probability does not always imply high revenue risk.

## Phase 2: Revenue-Weighted Risk Strategy

A revenue-adjusted risk score was created:

RiskScore = ChurnProbability × MonetaryValue

In [10]:
# Creating Revenue-Weighted Risk Score
results_df["RiskRevenueScore"] = (
    results_df["Churn_Prob"] * results_df["Monetary"]
)

results_df = results_df.sort_values(
    "RiskRevenueScore",
    ascending=False
)

high_risk_value = results_df.head(top_n)

In [12]:
# Recalculating Revenue at Risk
revenue_at_risk_value = high_risk_value["Monetary"].sum()

print("Revenue at Risk (Value-weighted):",
      revenue_at_risk_value)

print("Percentage of Revenue at Risk:",
      revenue_at_risk_value / total_revenue)

Revenue at Risk (Value-weighted): 1363137.4100000001
Percentage of Revenue at Risk: 0.41293871050222364


In [13]:
saved_revenue_value = revenue_at_risk_value * success_rate
campaign_cost_value = len(high_risk_value) * cost_per_customer

roi_value = (saved_revenue_value - campaign_cost_value) / campaign_cost_value

print("Saved Revenue:", saved_revenue_value)
print("Campaign Cost:", campaign_cost_value)
print("ROI:", roi_value)

Saved Revenue: 272627.482
Campaign Cost: 31400
ROI: 7.682403885350319


This approach:

- Increased revenue concentration within target segment
- Improved campaign ROI significantly


## Strategic Insight

Not all churn is equal.

Retention efforts should prioritize customers based on:
- Churn likelihood
- Revenue exposure

This integrates machine learning with financial decision-making.